# Chapitre 6 - Cours 2 : rééquilibrer les partitions

Un cluster ne garde pas la même taille toute sa vie : les données et le débit
grossissent, des machines tombent en panne et sont remplacées. Il faut alors **déplacer**
des données d'un nœud à l'autre : c'est le **rééquilibrage**. Ce notebook couvre la
section *Rebalancing Partitions* : *Strategies for Rebalancing* (*How not to do it: hash
mod N*, *Fixed number of partitions*, *Dynamic partitioning*, *Partitioning
proportionally to nodes*), l'encadré *Consistent Hashing* de la section *Partitioning by
Hash of Key*, et *Operations: Automatic or Manual Rebalancing*.

**Plan.**

1. Ce qu'on attend d'un rééquilibrage, et la borne inférieure du volume à déplacer.
2. Ce qu'il ne faut pas faire : haché modulo le nombre de nœuds.
3. Un nombre fixe de partitions, plus nombreuses que les nœuds.
4. Le partitionnement dynamique : des partitions qui se scindent.
5. Des partitions proportionnelles aux nœuds, et le hachage cohérent.
6. Rééquilibrage automatique ou manuel : le risque de cascade.

In [1]:
import time

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.patches import Wedge

from ddia import partitioning as P
from ddia.diagrams import Diagram
from ddia.plotting import (AMBER, GRAY, GREEN, INDIGO, LIGHT, ROSE, SKY, SLATE, TEAL, VIOLET, COLORS,
                           annotate_bars, finish, set_style, subtitle)

set_style()
plt.rcParams["axes.titlepad"] = 16
pd.set_option("display.precision", 3)
rng = np.random.default_rng(0)
fmt = lambda v: f"{v:,.0f}".replace(",", " ")
KEYS = np.arange(200_000)                       # 200 000 clés entières pour toutes les mesures